# Notebook 1: Identify Crop And Part With Router

Bu notebook sadece router yolunu calistirir: bootstrap, model erisim kontrolu, opsiyonel on-yukleme ve tek goruntu analizi.

Notebook 1 hastalikli goruntulerde dogru **bitki/crop yonlendirmesi** icin tasarlanmistir; hastalik sinifi tahmini yapmaz.

Router ayni runtime icinde bir kez hazirlanir ve sonraki denemelerde yeniden kullanilir. Boylece her yeni goruntu denemesinde modeller tekrar indirilmez veya yeniden yuklenmez.

Donen sonuc ust seviyede `crop` ve `part` alanlarini korur. Ayrica `router` blogu icinde router durumu, birincil tespit ve tespit sayisi bulunur. Bu notebook crop adapter yuklemez.

Bu surumde varsayilan router profili `balanced` olacak sekilde ayarlanmistir ve tanisal cikti (aday crop'lar, margin, unknown/rejection sinyalleri) gorunur hale getirilmistir. Bu tercih, acik-kume ve secmeli-reddetme literaturunde belirsiz durumda asiri guvenli etiket yerine daha kontrollu karar verilmesi geregiyle uyumludur (Hendrycks & Gimpel, 2017; Guo et al., 2017; Geifman & El-Yaniv, 2019).


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell01_bootstrap.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell02_access_check.py', globals())


In [ ]:
import json
from scripts.colab_repo_bootstrap import login_and_check_hf_token, running_in_colab
from scripts.colab_router_adapter_inference import clear_router_cache, ensure_router_ready, run_inference

# Notebook ayarlari. Yeni baslayan biri icin once sadece IMAGE_PATH / upload akisiyla ilerlemek yeterlidir.
NOTEBOOK_PROFILE = 'a100_colab_default'  # custom, a100_colab_default, cpu_debug

INFERENCE_PROFILES = {
    'custom': {},
    'a100_colab_default': {
        'CONFIG_ENV': 'colab',
        'DEVICE': 'cuda',
        'REQUIRE_HF_LOGIN': False,
        'CROP_HINT': None,
        'PART_HINT': None,
        'FORCE_UPLOAD_IF_NO_IMAGE': True,
        # Diseased goruntulerde daha guvenli crop secimi icin balanced onerilir.
        'ROUTER_RUNTIME_PROFILE': 'balanced',
    },
    'cpu_debug': {
        'CONFIG_ENV': 'colab',
        'DEVICE': 'cpu',
        'REQUIRE_HF_LOGIN': False,
        'CROP_HINT': None,
        'PART_HINT': None,
        'FORCE_UPLOAD_IF_NO_IMAGE': False,
        'ROUTER_RUNTIME_PROFILE': 'balanced',
    },
}

CONFIG_ENV = 'colab'
DEVICE = 'cuda'
REQUIRE_HF_LOGIN = False
CROP_HINT = None
PART_HINT = None
IMAGE_PATH = None  # Opsiyonel varsayilan goruntu yolu
FORCE_UPLOAD_IF_NO_IMAGE = True
PRELOAD_ROUTER = True
RESET_ROUTER_CACHE = False
ROUTER_RUNTIME_PROFILE = 'balanced'

# Router karar kalitesini izlemek icin tanisal cikti (onerilir: True).
SHOW_ROUTER_DIAGNOSTICS = True
TOP_CROP_CANDIDATES = 3
PRINT_JSON_RESULT = False

# Notebook-ici guvenlik kapisi: router-only tahmini belirsizse sonucu uncertain/unknown yapar.
ENFORCE_NOTEBOOK_ROUTER_GATE = True
ROUTER_RESULT_MIN_CONFIDENCE = 0.65
ROUTER_RESULT_MIN_MARGIN = 0.10

profile = dict(INFERENCE_PROFILES.get(NOTEBOOK_PROFILE, {}))
CONFIG_ENV = str(profile.get('CONFIG_ENV', CONFIG_ENV))
DEVICE = str(profile.get('DEVICE', DEVICE))
REQUIRE_HF_LOGIN = bool(profile.get('REQUIRE_HF_LOGIN', REQUIRE_HF_LOGIN))
CROP_HINT = profile.get('CROP_HINT', CROP_HINT)
PART_HINT = profile.get('PART_HINT', PART_HINT)
FORCE_UPLOAD_IF_NO_IMAGE = bool(profile.get('FORCE_UPLOAD_IF_NO_IMAGE', FORCE_UPLOAD_IF_NO_IMAGE))
PRELOAD_ROUTER = bool(profile.get('PRELOAD_ROUTER', PRELOAD_ROUTER))
RESET_ROUTER_CACHE = bool(profile.get('RESET_ROUTER_CACHE', RESET_ROUTER_CACHE))
ROUTER_RUNTIME_PROFILE = str(profile.get('ROUTER_RUNTIME_PROFILE', ROUTER_RUNTIME_PROFILE) or '').strip() or None

# Gerekirse son override katmani.
INFERENCE_OVERRIDES = {}
if INFERENCE_OVERRIDES:
    CONFIG_ENV = str(INFERENCE_OVERRIDES.get('CONFIG_ENV', CONFIG_ENV))
    DEVICE = str(INFERENCE_OVERRIDES.get('DEVICE', DEVICE))
    REQUIRE_HF_LOGIN = bool(INFERENCE_OVERRIDES.get('REQUIRE_HF_LOGIN', REQUIRE_HF_LOGIN))
    CROP_HINT = INFERENCE_OVERRIDES.get('CROP_HINT', CROP_HINT)
    PART_HINT = INFERENCE_OVERRIDES.get('PART_HINT', PART_HINT)
    IMAGE_PATH = INFERENCE_OVERRIDES.get('IMAGE_PATH', IMAGE_PATH)
    FORCE_UPLOAD_IF_NO_IMAGE = bool(INFERENCE_OVERRIDES.get('FORCE_UPLOAD_IF_NO_IMAGE', FORCE_UPLOAD_IF_NO_IMAGE))
    PRELOAD_ROUTER = bool(INFERENCE_OVERRIDES.get('PRELOAD_ROUTER', PRELOAD_ROUTER))
    RESET_ROUTER_CACHE = bool(INFERENCE_OVERRIDES.get('RESET_ROUTER_CACHE', RESET_ROUTER_CACHE))
    ROUTER_RUNTIME_PROFILE = str(INFERENCE_OVERRIDES.get('ROUTER_RUNTIME_PROFILE', ROUTER_RUNTIME_PROFILE) or '').strip() or None
    SHOW_ROUTER_DIAGNOSTICS = bool(INFERENCE_OVERRIDES.get('SHOW_ROUTER_DIAGNOSTICS', SHOW_ROUTER_DIAGNOSTICS))
    TOP_CROP_CANDIDATES = int(INFERENCE_OVERRIDES.get('TOP_CROP_CANDIDATES', TOP_CROP_CANDIDATES))
    PRINT_JSON_RESULT = bool(INFERENCE_OVERRIDES.get('PRINT_JSON_RESULT', PRINT_JSON_RESULT))
    ENFORCE_NOTEBOOK_ROUTER_GATE = bool(
        INFERENCE_OVERRIDES.get('ENFORCE_NOTEBOOK_ROUTER_GATE', ENFORCE_NOTEBOOK_ROUTER_GATE)
    )
    ROUTER_RESULT_MIN_CONFIDENCE = float(
        INFERENCE_OVERRIDES.get('ROUTER_RESULT_MIN_CONFIDENCE', ROUTER_RESULT_MIN_CONFIDENCE)
    )
    ROUTER_RESULT_MIN_MARGIN = float(
        INFERENCE_OVERRIDES.get('ROUTER_RESULT_MIN_MARGIN', ROUTER_RESULT_MIN_MARGIN)
    )

print(
    f'[AYAR] profil={NOTEBOOK_PROFILE} env={CONFIG_ENV} device={DEVICE} '
    f'crop_hint={CROP_HINT} part_hint={PART_HINT} preload_router={PRELOAD_ROUTER} '
    f'router_runtime_profile={ROUTER_RUNTIME_PROFILE} diagnostics={SHOW_ROUTER_DIAGNOSTICS} top_candidates={TOP_CROP_CANDIDATES} '
    f'notebook_gate={ENFORCE_NOTEBOOK_ROUTER_GATE} min_conf={ROUTER_RESULT_MIN_CONFIDENCE:.2f} min_margin={ROUTER_RESULT_MIN_MARGIN:.2f}'
)

HF_READY = False
if REQUIRE_HF_LOGIN:
    HF_READY = login_and_check_hf_token(print_fn=print)
    if not HF_READY:
        raise RuntimeError('Notebook ayari HF login istiyor ama dogrulama basarisiz oldu.')
else:
    print('[HF] Zorunlu login kapali. Gated model gerekiyorsa Colab secret ekleyin.')

if RESET_ROUTER_CACHE:
    clear_router_cache()
    print('[ROUTER] Oturum cache temizlendi.')

if CROP_HINT:
    print('[ROUTER] crop_hint verildi; router atlanacak, on-yukleme yapilmadi.')
elif PRELOAD_ROUTER:
    ensure_router_ready(
        config_env=CONFIG_ENV,
        device=DEVICE,
        status_printer=print,
        runtime_profile=ROUTER_RUNTIME_PROFILE,
    )
    print('[NOTEBOOK] Router bu runtime icin hazir. Sonraki hucreyi yeni bir goruntu ile tekrar calistirabilirsiniz.')
else:
    print('[NOTEBOOK] PRELOAD_ROUTER kapali. Ilk analiz cagrisi routeri bu sirada yukleyecek.')

In [ ]:
ANALYSIS_IMAGE_PATH = IMAGE_PATH  # Yeni dosya yolu verin veya None birakip yukleme kullanin.
UPLOAD_NEW_IMAGE = ANALYSIS_IMAGE_PATH is None and FORCE_UPLOAD_IF_NO_IMAGE

if ANALYSIS_IMAGE_PATH is None and UPLOAD_NEW_IMAGE:
    if not running_in_colab():
        raise ValueError('Colab disinda calisiyorsaniz ANALYSIS_IMAGE_PATH degerini elle verin.')
    from google.colab import files
    uploaded = files.upload()
    ANALYSIS_IMAGE_PATH = next(iter(uploaded.keys()))

if ANALYSIS_IMAGE_PATH is None:
    raise ValueError('Yukleme kapaliyken ANALYSIS_IMAGE_PATH zorunludur.')

# Defer PIL/matplotlib imports until needed
from PIL import Image, ImageDraw

analysis_image = Image.open(ANALYSIS_IMAGE_PATH).convert('RGB')

# Router cache kullanildigi icin farkli bir goruntu denemek icin bu hucreyi tekrar calistirmaniz yeterlidir.
result = run_inference(
    ANALYSIS_IMAGE_PATH,
    config_env=CONFIG_ENV,
    crop_hint=CROP_HINT,
    part_hint=PART_HINT,
    device=DEVICE,
    status_printer=print,
    include_diagnostics=SHOW_ROUTER_DIAGNOSTICS,
    top_candidates=TOP_CROP_CANDIDATES,
    runtime_profile=ROUTER_RUNTIME_PROFILE,
)

runtime_profile = str(result.get('runtime_profile', '') or '')
if runtime_profile:
    print(f'[ROUTER] active_runtime_profile={runtime_profile}')

adapter_target = dict(result.get('adapter_target') or {})
if adapter_target.get('crop'):
    print(
        f"[ADAPTER] hedef_crop={adapter_target.get('crop')} "
        f"adapter_dir={adapter_target.get('adapter_dir')} "
        f"exists={bool(adapter_target.get('exists', False))}"
    )

router_details = dict(result.get('router_details') or {})
detections = list(router_details.get('detections') or [])
annotated_image = analysis_image.copy()
draw = ImageDraw.Draw(annotated_image)
width, height = annotated_image.size
box_count = 0

for index, detection in enumerate(detections, start=1):
    bbox = detection.get('bbox')
    if not bbox or len(bbox) < 4:
        continue
    x1, y1, x2, y2 = [float(value) for value in bbox[:4]]
    if max(abs(x1), abs(y1), abs(x2), abs(y2)) <= 1.5:
        x1 *= width
        x2 *= width
        y1 *= height
        y2 *= height
    x1 = max(0.0, min(width - 1.0, x1))
    y1 = max(0.0, min(height - 1.0, y1))
    x2 = max(0.0, min(width - 1.0, x2))
    y2 = max(0.0, min(height - 1.0, y2))
    if x2 <= x1 or y2 <= y1:
        continue
    box_count += 1
    color = (255, 99, 71) if index == 1 else (66, 133, 244)
    draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
    label = f"{index}: {str(detection.get('crop', 'unknown') or 'unknown')} / {str(detection.get('part', 'unknown') or 'unknown')}"
    text_x = x1 + 4
    text_y = max(0.0, y1 - 18)
    draw.rectangle([text_x - 2, text_y - 2, text_x + 230, text_y + 16], fill=(0, 0, 0))
    draw.text((text_x, text_y), label, fill=(255, 255, 255))

diagnostics = dict(result.get('diagnostics') or {})
top_candidates = list(diagnostics.get('top_crop_candidates') or [])
if top_candidates:
    print('[TANISAL] En iyi crop adaylari:')
    for index, candidate in enumerate(top_candidates, start=1):
        crop_name = str(candidate.get('crop', 'unknown') or 'unknown')
        part_name = str(candidate.get('part', 'unknown') or 'unknown')
        crop_conf = float(candidate.get('crop_confidence', 0.0) or 0.0)
        part_conf = float(candidate.get('part_confidence', 0.0) or 0.0)
        print(
            f"  {index}. crop={crop_name} part={part_name} "
            f"crop_conf={crop_conf:.3f} part_conf={part_conf:.3f}"
        )

if diagnostics:
    print(
        f"[TANISAL] crop_margin={float(diagnostics.get('crop_confidence_margin', 0.0) or 0.0):.3f} "
        f"raw_part_label={str(diagnostics.get('raw_part_label', '') or '')} "
        f"raw_part_conf={float(diagnostics.get('raw_part_confidence', 0.0) or 0.0):.3f} "
        f"unknown_conf={float(diagnostics.get('part_unknown_confidence', 0.0) or 0.0):.3f}"
    )
    rejection_reason = str(diagnostics.get('part_rejection_reason', '') or '').strip()
    if rejection_reason:
        print(f'[TANISAL] part_rejection_reason={rejection_reason}')

primary_crop_conf = float(result.get('router_confidence', 0.0) or 0.0)
runner_up_crop_conf = None
if len(top_candidates) > 1:
    runner_up_crop_conf = float(top_candidates[1].get('crop_confidence', 0.0) or 0.0)
effective_margin = None if runner_up_crop_conf is None else (primary_crop_conf - runner_up_crop_conf)

notebook_gate_reasons = []
if ENFORCE_NOTEBOOK_ROUTER_GATE and not CROP_HINT:
    if primary_crop_conf < float(ROUTER_RESULT_MIN_CONFIDENCE):
        notebook_gate_reasons.append(
            f"crop_confidence={primary_crop_conf:.3f} < min_confidence={float(ROUTER_RESULT_MIN_CONFIDENCE):.3f}"
        )
    if effective_margin is not None and effective_margin < float(ROUTER_RESULT_MIN_MARGIN):
        notebook_gate_reasons.append(
            f"crop_margin={effective_margin:.3f} < min_margin={float(ROUTER_RESULT_MIN_MARGIN):.3f}"
        )

if notebook_gate_reasons:
    notebook_gate_message = 'Notebook router gate rejected prediction: ' + '; '.join(notebook_gate_reasons)
    print(f'[GATE] {notebook_gate_message}')
    result['status'] = 'uncertain'
    result['message'] = notebook_gate_message
    result['crop'] = 'unknown'
    result['part'] = 'unknown'
    result['adapter_target'] = {'crop': None, 'adapter_dir': None, 'exists': False}
    result['notebook_gate'] = {
        'applied': True,
        'accepted': False,
        'reasons': notebook_gate_reasons,
        'min_confidence': float(ROUTER_RESULT_MIN_CONFIDENCE),
        'min_margin': float(ROUTER_RESULT_MIN_MARGIN),
    }
elif ENFORCE_NOTEBOOK_ROUTER_GATE:
    result['notebook_gate'] = {
        'applied': True,
        'accepted': True,
        'reasons': [],
        'min_confidence': float(ROUTER_RESULT_MIN_CONFIDENCE),
        'min_margin': float(ROUTER_RESULT_MIN_MARGIN),
    }

if box_count <= 0:
    print('[TANISAL] Router detection bbox bulunamadi; sadece ham goruntu gosteriliyor.')

if PRINT_JSON_RESULT:
    print(json.dumps(result, indent=2))

# Defer matplotlib import until visualization is needed
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2 if box_count > 0 else 1, figsize=(16, 8))
if box_count > 0:
    axes[0].imshow(analysis_image)
    axes[0].set_title('Orijinal')
    axes[0].axis('off')
    axes[1].imshow(annotated_image)
    axes[1].set_title('Router kutulari')
    axes[1].axis('off')
else:
    axes.imshow(analysis_image)
    axes.set_title('Orijinal')
    axes.axis('off')
plt.tight_layout()

result

In [ ]:
ROUTER_EVAL_ROOT = None  # Ornek: '/content/router_part_eval'
ROUTER_EVAL_OUTPUT = None  # Ornek: '/content/router_part_eval_summary.json'
RUN_ROUTER_EVAL = False
EVAL_MIN_CONFIDENCE_GRID = '0.20,0.25,0.30,0.35,0.40,0.45,0.50'
EVAL_MARGIN_GRID = '0.00,0.02,0.05,0.08,0.10,0.12,0.15'

if RUN_ROUTER_EVAL and ROUTER_EVAL_ROOT:
    from pathlib import Path
    from scripts.evaluate_router_part_surface import evaluate_router_part_surface, _parse_grid

    eval_result = evaluate_router_part_surface(
        Path(ROUTER_EVAL_ROOT),
        config_env=CONFIG_ENV,
        device=DEVICE,
        min_confidence_grid=_parse_grid(EVAL_MIN_CONFIDENCE_GRID),
        margin_grid=_parse_grid(EVAL_MARGIN_GRID),
    )
    if ROUTER_EVAL_OUTPUT:
        output_path = Path(ROUTER_EVAL_OUTPUT)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(json.dumps(eval_result, indent=2), encoding='utf-8')
        print(f'[EVAL] wrote {output_path}')

    print(f"[EVAL] dataset_root={eval_result['dataset_root']}")
    print(f"[EVAL] sample_count={eval_result['sample_count']} crop_accuracy={eval_result['crop_accuracy']:.4f}")
    for crop_name, crop_summary in sorted((eval_result.get('crops') or {}).items()):
        print(
            f"[EVAL] crop={crop_name} sample_count={crop_summary.get('sample_count', 0)} "
            f"crop_accuracy={float(crop_summary.get('crop_accuracy', 0.0) or 0.0):.4f} "
            f"non_unknown_precision={float(crop_summary.get('non_unknown_precision', 0.0) or 0.0):.4f} "
            f"abstention_rate={float(crop_summary.get('abstention_rate', 0.0) or 0.0):.4f} "
            f"unsupported_part_emission={int(crop_summary.get('unsupported_part_emission', 0) or 0)}"
        )
    threshold_sweep = dict(eval_result.get('threshold_sweep') or {})
    recommended = dict(threshold_sweep.get('recommended') or {})
    if recommended:
        print(
            f"[EVAL] recommended min_confidence={float(recommended.get('min_confidence', 0.0) or 0.0):.3f} "
            f"margin={float(recommended.get('margin', 0.0) or 0.0):.3f} "
            f"precision={float(recommended.get('non_unknown_precision', 0.0) or 0.0):.4f} "
            f"abstention={float(recommended.get('abstention_rate', 0.0) or 0.0):.4f}"
        )
else:
    print('[EVAL] RUN_ROUTER_EVAL kapali veya ROUTER_EVAL_ROOT ayarlanmadi.')